# 2. Optimize a grid and re-evaluate its exported layout

## Before running

Use a source checkout. From its root, prepare a Python environment and install:

```sh
python -m pip install -e '.[notebooks]'
python scripts/bootstrap.py --jobs 4
python scripts/build_liblsqecc_adapter.py --source .deps/liblsqecc --build .deps/liblsqecc/build --out .deps/adapter
jupyter lab notebooks
```

The native build requires Git, CMake, a C++20 compiler and liblsqecc development
libraries; see [the integration guide](../docs/integration.md). The adapter is our
executable around liblsqecc that accepts custom layouts. Build it once, then reuse
it. Choose the kernel belonging to the environment where you installed the package.
Use **Restart Kernel and Run All**. Each run creates a separate evidence directory.

These small examples teach integration, not algorithm performance. All compilation
uses **wave scheduling with local instruction decomposition enabled**. Area means
all cells in the rectangular grid; slices are completed compiler frames; their
product is measured in tile-slices, not physical qubit-cycles. `M` cells are abstract
magic-state supplies, with compiler production timing fixed at 10.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import json
import subprocess

# Start Jupyter from the repository root or its notebooks directory.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "src/lattice_floorplan").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository checkout.")

# Change these two paths if you built the adapter somewhere else.
ADAPTER = ROOT / ".deps/adapter/liblsqecc_adapter"
EXPORTER = ROOT / ".deps/adapter/export_layout"
for executable in (ADAPTER, EXPORTER):
    if not executable.is_file():
        raise FileNotFoundError(f"Build the adapter first; missing {executable}")

from lattice_floorplan.compiler import Compiler, CompilerConfig
from lattice_floorplan.geometry import GridState, GeometryOracle
from lattice_floorplan.layout import Layout

# This example is already synthesized into Clifford+T gates.
qasm = (ROOT / "docs/_static/example.qasm").read_text()
compiler = Compiler(ADAPTER, CompilerConfig(local=True, timeout=30))
(ROOT / "runs").mkdir(exist_ok=True)
RUN = Path(mkdtemp(prefix="notebook-", dir=ROOT / "runs"))
print("Evidence directory:", RUN.relative_to(ROOT))


## Run the supplied optimizer

`optimize_layout` accepts circuit text, a `GridState`, the adapter location and a
new output directory. Here `sa` means simulated annealing. The request budget
includes failed candidates and cached results, not just compiler invocations.
The function saves the winning grid, logical identities and matched circuit,
then recompiles that exported pair before returning.


In [ ]:
from lattice_floorplan.api import optimize_layout
from lattice_floorplan.tempering import TemperingConfig

initial = GridState.parse(subprocess.run(
    [str(EXPORTER), "edpc", "5"], check=True, capture_output=True, text=True
).stdout)
result = optimize_layout(
    qasm, initial, adapter_path=ADAPTER, output_directory=RUN / "optimized",
    method="sa", budget=32, seed=61,
    search_config=TemperingConfig(move_supply=True),
    compiler_config=CompilerConfig(local=True, timeout=30),
)
print({key: result[key] for key in ("best_area", "best_slices", "best_objective")})


## Load a saved layout correctly

This section also works with a bundle produced by another optimizer that uses our
export function. Set `bundle` to that directory. Use the **original circuit** and
saved logical positions with `GeometryOracle`: it handles circuit relabeling for
you. Passing the already mapped circuit would relabel it twice. The fresh evidence
folder below avoids reusing the optimizer's cached scores.


A **bundle** is an ordinary layout output folder, not a special file format. It
contains `layout.txt`, `layout.svg`, `identity.json`, `original-circuit.qasm`,
`mapped-circuit.qasm` and `resources.json`. Keep them together: the grid alone
does not record the logical-qubit identities.


In [ ]:
bundle = RUN / "optimized"  # Replace with your own exported bundle.
original_qasm = (bundle / "original-circuit.qasm").read_text()
identity = json.loads((bundle / "identity.json").read_text())
winning_grid = GridState.parse((bundle / "layout.txt").read_text(),
                               identity["logical_qubit_positions"])
evaluator = GeometryOracle(compiler, RUN / "independent-evaluation")
measurement = evaluator.evaluate(original_qasm, winning_grid)
if measurement["status"] != "success":
    raise RuntimeError(measurement)
print("Area (tiles):", winning_grid.area)
print("Time slices:", measurement["slices"])
print("Volume (tile-slices):", winning_grid.area * measurement["slices"])
assert measurement["slices"] == result["best_slices"]


The returned score is a compiler measurement, not a prediction from the optimizer.
Keep `layout.txt`, `identity.json`, both circuit files and the evidence together.
`layout.svg` in the bundle visualizes the selected grid. Grid dimensions and initial
qubit assignments may change during this search; the supply count remains fixed.
